In [ ]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import SGDClassifier, LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
import scipy

In [ ]:
dataset_druggable = pd.read_csv("../input/combined_DepMap_21Q3_druggable.csv")
dataset_ccle = pd.read_csv("../input/combined_DepMap_21Q3_CCLE_expression.csv")
druggable_X = dataset_druggable.iloc[:, 1:-4686]
druggable_y = dataset_druggable.iloc[:, -4686:]
ccle_X = dataset_ccle.iloc[:, 1:-4686]
ccle = dataset_ccle.iloc[:, -4686:]


In [ ]:
dataset = pd.read_csv("../input/combined_DepMap_21Q3.csv")
num_gene = 17651
drug_X = dataset.iloc[:, 1:num_gene+1]
drug_y = dataset.iloc[:, -4686:]
drug_list = drug_y.columns.tolist()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import numpy as np
import os
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.feature_selection import *
from optuna.integration import OptunaSearchCV
import optuna
import imblearn
import xgboost as xgb
import cupy
from cuml import RandomForestClassifier as cuRF
import cudf

In [ ]:
drug_y['BRD-A17300716-001-01-5::2.5::HTS'].sum()/len(drug_y['BRD-A17300716-001-01-5::2.5::HTS'])

In [ ]:
sgd = joblib.load('/users/ysu13/repos/drug_sensitivity_mlpred/output/pensive_blackburn/BRD-A25004090-001-08-4::2.5::HTS/SGDClassifier_BRD-A25004090-001-08-4::2.5::HTS.joblib')

In [ ]:
y_pred_proba= sgd.predict_proba(X_test)[:,1]
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

In [ ]:
y_pred_proba

In [ ]:
average_precision_score(y_test, y_pred_proba)

In [ ]:
from decimal import Decimal
'%.3E' % Decimal(max(precision[:-1]))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(drug_X, drug_y['BRD-A25004090-001-08-4::2.5::HTS'], test_size=0.2, random_state=20)

In [ ]:
X_train_cu = cudf.DataFrame(X_train)
y_train_cu = cudf.Series(y_train)

In [ ]:
dtrain = xgb.DMatrix(X_train.values, label=y_train.values)

In [ ]:
X_train_cp = cupy.array(X_train.values)
y_train_cp = cupy.array(y_train.values)

In [ ]:
clf = Pipeline([
  ('sampling', SMOTE(random_state=72)),
  ('classifier', XGBClassifier(device='cuda', tree_method = 'hist'))
])


In [ ]:
xgb_cpu = Pipeline([('classifier', XGBClassifier( n_jobs = 8, max_depth = 5))])
xgb_cpu.fit(X_train, y_train)

In [ ]:
xgb_gpu = Pipeline([('classifier', XGBClassifier(  max_depth = 5, device='cuda', tree_method = 'hist'))])
xgb_gpu.fit(X_train, y_train)

In [ ]:
clf.fit(X_train , y_train)

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [ ]:
cu_rf = Pipeline([('classifier', cuRF())])

In [ ]:

skl_rf = Pipeline([('classifier', RandomForestClassifier(n_jobs = 10))])


In [ ]:
sgd = Pipeline([('scaler', StandardScaler(with_mean=False)),
		 ('classifier',SGDClassifier(penalty = 'elasticnet', loss = 'log_loss', n_jobs = 20, class_weight='balanced'))])
search_params = {
   'l1_ratio': optuna.distributions.FloatDistribution(0.0, 0.2),
    'alpha': optuna.distributions.FloatDistribution(5e-3, 5e-2, log=True),
    'max_iter': optuna.distributions.IntDistribution(1000, 2500)
}
optuna_search_params = {
 	    "max_depth" : optuna.distributions.IntDistribution(5, 20),
 	    "n_estimators" : optuna.distributions.IntDistribution(50, 200),
    }
search_params= {'classifier__' + key: search_params[key] for key in search_params}
optuna_search_params = {'classifier__' + key: optuna_search_params[key] for key in optuna_search_params}
search_estimator_best = OptunaSearchCV(
            xgb_cpu,
            param_distributions=optuna_search_params,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring='f1',
            n_jobs=4,
            n_trials=20,
            verbose=2
)

In [ ]:
search_estimator_best.fit(X_train, y_train)
best= search_estimator_best.best_estimator_


In [ ]:
search_estimator_best.fit(X_train, y_train)
best= search_estimator_best.best_estimator_


In [ ]:
y_pred = best.predict_proba(X_test.values)[:,1]
y_pred0 = best.predict(X_test)
precision, recall, thresholds = precision_recall_curve(y_test, y_pred)
aupr_score = average_precision_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

In [ ]:
best.predict_proba(X_test.values)

In [ ]:
sgd2 = SGDClassifier(penalty = 'elasticnet', random_state=777, loss = 'log_loss', n_jobs = 20, alpha = 0.01, l1_ratio = 0.2, max_iter=5000,class_weight='balanced')
sgd2.fit(X_train, y_train)

In [ ]:
plt.hist(best['classifier'].coef_[0], bins =100)
plt.show()

In [ ]:
pd.DataFrame({'feature': list(X_train.columns),
					'importances': clf['classification'].feature_importances_ * 100}).\
					 sort_values('importances', ascending = False)

In [ ]:
pd.DataFrame({'feature': list(X_train.columns),
					'importances': xgb2['classifier'].feature_importances_ * 100}).\
					 sort_values('importances', ascending = False)

In [ ]:
y_pred = sgd2.predict_proba(X_train)[:,1]
precision, recall, thresholds = precision_recall_curve(y_train, y_pred)
aupr_score = average_precision_score(y_train, y_pred)
fpr, tpr, _ = roc_curve(y_train, y_pred)
roc_auc = auc(fpr, tpr)

In [ ]:
from sklearn_evaluation.plot import confusion_matrix, ConfusionMatrix

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")

In [ ]:
a.ax_=

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color='blue', lw=2, label=f'AUPR curve (area = {aupr_score:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc="lower left")
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.show()

In [ ]:
%timeit cross_validate(xgb, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=1)

In [ ]:


%timeit cross_validate(xgb1, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=20)


In [ ]:
xgb2 = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 4 )
%timeit cross_validate(xgb2, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=5)


In [ ]:
xgb3 = XGBClassifier(learning_rate= 0.07297156269889893, max_depth=6, min_child_weight= 1, subsample=0.8042517018984596, n_jobs = 15 )
%timeit cross_validate(xgb3, X_train, y_train['BRD-A00077618-236-07-6::2.5::HTS'], scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=5)


In [ ]:
from sklearn.metrics import get_scorer
for s in ['balanced_accuracy', 'precision', 'recall', 'f1', 'average_precision', 'roc_auc']:
    score_ = get_scorer(s)
    print(score_(xgb3, X_test, y_test))

In [ ]:
xgb3.fit(X_train, y_train)

In [ ]:
score_(xgb3, X_test, y_test)

In [ ]:
# SOME PARAMETERS FOR GRIDSEARCH
GRID_SEARCH_PARAM = {
'XGB':{
 	"learning_rate" : [0.05,0.10,0.15,0.20],
 	"max_depth" : [ 3, 4, 5, 6, 8, 10, 12, 15],
 	"min_child_weight" : [ 1, 3, 5, 7 ],
 	"gamma": [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
	 "n_estimators": [50, 100, 200]
},
 'RF':{
	 "n_estimators": [100, 150, 250],
	 "max_depth" : [20, 50, 100, 200],
 },
 'SGD':{
	 'l1_ratio':[0.2, 0.15, 0.1, 0.05],
	 'alpha':[0.02, 0.05]
 }
}

# Basic function to handel sklearn and traditional model training and basic hyperparameter optimization
# TODO refactor this into a class maybe, class DrugModel
def skl_drug_model(X, Y, drug, model = XGBClassifier, oversample = True, fixed_params={}, search_params = {}, search_method = 'gridcv'):

	assert model in [XGBClassifier, RandomForestClassifier, SGDClassifier, LGBMClassifier], f' {model} type not supported'

	
	y = Y[drug]

	 # split X and y into training and testing sets
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


	if oversample:
		ros = SMOTE(random_state=72)
		X_res, y_res = ros.fit_resample(X_train, y_train)
		oversample = 'oversample'
	else:
		X_res, y_res = X_train, y_train
		oversample = ''


	 # instantiate the classifier
	 

	# k-fold cross validation using multiple metric evaluation
	kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
	 #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
	if oversample:
		imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
							  ('classifier', model(n_jobs=10, **fixed_params))])
		grid_search_parameters = {'classifier__' + key: search_params[key] for key in search_params}
	else:
		imba_pipeline = model(**fixed_params, n_jobs=20)
		grid_search_parameters = search_params
	 #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
	model0 = model(**fixed_params, n_jobs=20)

	 #perform gridsearch if needed
	if search_method == 'gridcv' and search_params:
		grid_imba = HalvingRandomSearchCV(imba_pipeline, param_distributions=grid_search_parameters, cv=kfold, scoring='precision')
		#grid_imba.fit(X_train, y_train) 
		cv_results = cross_validate(grid_imba, X, y, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold,  n_jobs=10)
		print(cv_results)
		print(grid_imba)
		#return cv_results, grid_imba
		#best_params = {key.removeprefix('classifier__'):grid_imba.best_params_[key] for key in grid_imba.best_params_}
		#print(best_params)
		#model0.set_params(**best_params)
	else:
		cv_results = cross_validate(model0, X_train, y_train, scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], cv=kfold, njobs = 10)
		grid_imba = model0
	 
	if oversample:
		imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
							  ('classifier', model0)])
	else:
		imba_pipeline = model0
		  
	# generate cross validation results of best model with correct oversampling 
	# OVERSAMPLING must come after validation split for correct validation , thus the use of pipeline
	# cross_validate function will first split into train/validate, then feed training data into pipeline (oversampling + training)
	#cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 20)
	cv_results = pd.DataFrame(cv_results)
	print(cv_results)

	 # declare parameters
	 

	 # fit the classifier to the training data
	model0.fit(X_res, y_res)

	 # save the trained model
	
		  
	final_results = {
		'best_model': model0,
		'drug': drug,
		'model_class': model,
		'model_search': grid_imba,
		'search_method': search_method,
		'X_test': X_test,
		'Y_test': y_test,
		'X_train': X_train,
		'Y_train': y_train,
		'cv_results':cv_results,
		'oversample': True if oversample else False
	}

	return final_results


def write_drug_model_result(model_results, out_dir):
	 
	model0 = model_results['best_model']
	oversample = 'oversample' if model_results['oversample'] else ''
	drug = model_results['drug']
	model_class = model_results['model_class']

	model_name = {
		  XGBClassifier:'XGBClassifier',
		  RandomForestClassifier:'RandomForestClassifier',
		  SGDClassifier:'SGDClassifier',
		  LGBMClassifier:'LGBMClassifier'

	}[model_class]

	full_out_dir = f'{out_dir}/{oversample}/{drug}/'
	X_test, y_test = model_results['X_test'], model_results['Y_test']
	X_train, y_train = model_results['X_train'], model_results['Y_train']
	y_pred = model0.predict(X_test)

	os.makedirs(full_out_dir, exist_ok=True)

	joblib.dump(model0, f'{full_out_dir}/{model_name}_{drug}.joblib')
	
	model_results['cv_results'].to_csv(f'{full_out_dir}/{model_name}_cv_results_{drug}.csv')

	print(drug,
		 f'{model_class}_model_parameters', model0, "\n",
		 "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
		 file=open(f'{full_out_dir}/{model_name}_confusion_matrix.txt', "a"))

	model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
	model_report = pd.DataFrame(model_report).transpose()
	
	if (model_report.index == "1").any() == True:
		r1 = pd.DataFrame(model_report.loc["1"]).transpose()
		r1.to_csv(f'{full_out_dir}/{model_name}_classification_report_{drug}.csv')
	else:
		print(drug, "Nothing predicted as 1",
			   file=open(f'{out_dir}/{oversample}/{model_name}_classification_report_log.txt', "a"))

	 

	 # feature importance with XGBoost
	if model_class in [XGBClassifier, RandomForestClassifier]:
		fi = pd.DataFrame({'feature': list(X_train.columns),
					'importances': model0.feature_importances_ * 100}).\
					 sort_values('importances', ascending = False)
		fi.to_csv(f'{full_out_dir}/{model_name}_feature_importance_{drug}.csv')

		# plot feature importance
		plt.figure(figsize=(10, 8))
		sns.barplot(x='importances', y='feature', data=fi.head(20))
		plt.title(f'{model_name} Feature Importance for {drug}')
		plt.tight_layout()
		plt.savefig(f'{full_out_dir}/{model_name}_feature_importance_{drug}.png')
		plt.close()
	# save the model report
	model_report.to_csv(f'{full_out_dir}/{model_name}_classification_report_{drug}.csv')
	print(f"Model for {drug} saved to {full_out_dir}/{model_name}_{drug}.joblib")
	return model_results




In [ ]:
model_map = {}
for d in drug_list[0:1]:
	cv_res1, grid_imba1 = skl_drug_model(drug_X, drug_y, d, 
		model = XGBClassifier, 
		oversample = False, 
		fixed_params={},
		search_params=GRID_SEARCH_PARAM['XGB']
	)


In [ ]:
model_map = {}
for d in drug_list[0:1]:
	model_map[d] = skl_drug_model(drug_X, drug_y, d, 
		model = LGBMClassifier, 
		oversample = False,
		search_params={},
		fixed_params={}
		)

In [ ]:
write_drug_model_result(model_map['BRD-A00077618-236-07-6::2.5::HTS'], out_dir = 'output')

In [ ]:
starttime = time.time()
grid_search = skl_drug_model('BRD-A00100033-001-08-9::2.5::HTS',   oversample = True, model = RandomForestClassifier
)
endtime = time.time()